# Telco Customer Churn Analysis - Portfolio Summary

**Business Context:** Analysis for telecom retention team to understand churn drivers, quantify revenue impact, and develop actionable risk segmentation for customer outreach.

**Key Findings:**
- 26.5% baseline churn rate represents **$1.67M annual revenue loss**
- Contract type is the dominant predictor (month-to-month customers churn at 42.7% vs 2.8% for two-year)
- First 12 months are the critical retention window (47.4% churn vs 17.1% after year one)
- Risk segmentation framework identifies 2,598 high-risk customers representing $100K+ monthly revenue exposure

**Analysis Approach:**
1. Exploratory analysis to identify key churn drivers
2. Predictive modeling with class balancing (80% recall achieved)
3. Customer behavioral profiling
4. Rule-based risk segmentation framework
5. Revenue impact quantification and executive reporting

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Install scikit-learn if needed
try:
    import sklearn
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn", "-q"])

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
# Load the dataset
df = pd.read_csv('../data/telco_churn.csv')

# Display basic info
print(f"Dataset Shape: {df.shape}")
print(f"\nChurn Rate: {(df['Churn'].value_counts(normalize=True)['Yes']*100):.1f}%")
df.head()

## Data Quality & Preparation

In [ ]:
# Fix data type issue - TotalCharges imported as object
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check for missing values
missing_count = df['TotalCharges'].isna().sum()
print(f"Missing TotalCharges: {missing_count} rows ({missing_count/len(df)*100:.2f}%)")

# Examine missing rows
print("\nMissing rows analysis:")
print(df[df['TotalCharges'].isna()][['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].describe())

**Finding:** Missing TotalCharges rows are new customers (tenure=0) who haven't been charged yet. All show 0% churn. Imputing with 0 to preserve this behaviorally distinct cohort.

In [ ]:
# Impute missing TotalCharges with 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Create binary churn flag for calculations
df['Churn_Flag'] = (df['Churn'] == 'Yes').astype(int)

---

## Exploratory Analysis: Key Churn Drivers

### Contract Type - Strongest Predictor

In [ ]:
# Contract type distribution by churn
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Contract', hue='Churn')
plt.title('Contract Type Distribution by Churn', fontweight='bold')
plt.xlabel('Contract Type')
plt.ylabel('Count')
plt.legend(title='Churn', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

# Calculate churn rates
contract_churn = df.groupby('Contract')['Churn_Flag'].agg(['sum', 'count', 'mean'])
contract_churn.columns = ['Churned', 'Total', 'Churn_Rate']
contract_churn['Churn_Rate_%'] = (contract_churn['Churn_Rate'] * 100).round(1)
contract_churn[['Total', 'Churned', 'Churn_Rate_%']]

**Key Finding:** Month-to-month contracts show 15x higher churn than two-year contracts. This is the dominant retention lever.

### Monthly Charges Distribution

In [ ]:
# Visualizing Monthly Charges Distribution
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='MonthlyCharges', multiple='stack', hue='Churn', bins=30)
plt.title('Monthly Charges Distribution by Churn', fontweight='bold')
plt.xlabel('Monthly Charges ($)')
plt.ylabel('Count')
plt.legend(title='Churn', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

# Summary stats
print("Monthly Charges by Churn Status:")
df.groupby('Churn')['MonthlyCharges'].describe().round(2)

**Finding:** Churned customers pay 21% more per month on average ($74 vs $61), suggesting a value perception issue.

### Tenure Distribution

In [ ]:
# Visualizing Tenure Distribution
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='tenure', multiple='stack', hue='Churn', bins=30)
plt.title('Tenure Distribution by Churn', fontweight='bold')
plt.xlabel('Tenure (months)')
plt.ylabel('Count')
plt.legend(title='Churn', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

# Tenure stats by churn status
print("Tenure Statistics by Churn:")
df.groupby('Churn')['tenure'].describe().round(2)

**Finding:** Bimodal distribution with concentration at extremes. Churned customers have half the tenure (18 months vs 38 months average), highlighting early-stage retention challenges.

---

## Predictive Modeling

### Model Setup

Selected features based on business logic and exploratory analysis:
- **Contract, InternetService, PaymentMethod** (categorical)
- **tenure, MonthlyCharges, TotalCharges** (numerical)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc

# Features for modeling
X = df[['Contract', 'InternetService', 'PaymentMethod', 'tenure', 'MonthlyCharges', 'TotalCharges']]
y = df['Churn']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['tenure', 'MonthlyCharges', 'TotalCharges']),
        ('cat', OneHotEncoder(drop='first', sparse_output=False),
         ['Contract', 'InternetService', 'PaymentMethod'])
    ])

# Create pipeline with class balancing
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

# Fit the model
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("Model trained successfully with class balancing")

**Note on Class Balancing:** Dataset has 3:1 imbalance (more non-churners). Using `class_weight='balanced'` to prioritize recall - better to flag potential churners than miss them.

### Model Performance

In [ ]:
# Classification report
print(classification_report(y_test, y_pred))

# Confusion matrix visualization
cm = confusion_matrix(y_test, y_pred, labels=['No', 'Yes'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.title('Confusion Matrix - Logistic Regression (Class Balanced)', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# Calculate key metrics
tn, fp, fn, tp = cm.ravel()
recall = tp / (tp + fn)
print(f"\n✓ Model achieves {recall*100:.0f}% recall on churners")
print(f"✓ Correctly identifies {tp} out of {tp+fn} actual churners in test set")

**Performance Summary:**
- **Recall: 80%** - Catches 4 out of 5 customers who will actually churn
- **Precision: 50%** - Half of flagged customers will actually churn
- **Trade-off justified:** For retention campaigns, false positives (wasted outreach) cost less than false negatives (lost customers)

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba, pos_label='Yes')
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve - Model Performance', fontweight='bold')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**AUC = 0.85** indicates strong discriminative ability. Model correctly ranks a random churner higher than a random non-churner 85% of the time.

---

## Behavioral Profiling: What Do Churners Look Like?

### Create Service Engagement Feature

In [ ]:
# Count add-on services
service_features = ['OnlineSecurity', 'TechSupport', 'OnlineBackup',
                   'DeviceProtection', 'StreamingTV', 'StreamingMovies']

df['Service_Count'] = df[service_features].apply(lambda row: (row == 'Yes').sum(), axis=1)

print("Service adoption vs churn:")
df.groupby('Service_Count').agg({
    'Churn_Flag': ['count', 'sum', 'mean']
}).round(3)

### Churner vs Retained: Numerical Comparison

In [ ]:
# Compare key metrics between churned and retained customers
profile_cols = ['MonthlyCharges', 'tenure', 'TotalCharges', 'Service_Count']

churn_profile = df.groupby('Churn')[profile_cols].mean().round(2).T
churn_profile.columns = ['Retained', 'Churned']
churn_profile['Difference_%'] = ((churn_profile['Churned'] - churn_profile['Retained'])
                                   / churn_profile['Retained'] * 100).round(1)
churn_profile

**Churner Profile:**
- **Tenure:** 50% shorter (18 months vs 38 months)
- **Monthly Charges:** 21% higher ($74 vs $61)
- **Service Count:** 18% lower (1.8 vs 2.2 services)
- **Total Charges:** Lower accumulated value due to shorter tenure

**Insight:** Churners pay more per month but receive fewer services - clear value perception gap.

### High-Risk Combinations: Billing & Payment

In [ ]:
# Identify risky billing/payment combinations
billing_churn = (df.groupby(['PaperlessBilling', 'PaymentMethod'])
                   .agg(
                       Customers=('customerID', 'count'),
                       Churn_Rate=('Churn_Flag', 'mean')
                   )
                   .round(3)
                   .sort_values('Churn_Rate', ascending=False)
                   .head(10))
billing_churn['Churn_Rate_%'] = (billing_churn['Churn_Rate'] * 100).round(1)
billing_churn[['Customers', 'Churn_Rate_%']]

**Finding:** Paperless billing + electronic check shows highest churn risk (~45%). This combination may signal lower engagement or payment friction.

### Monthly Charges Distribution - Visual Comparison

In [ ]:
# Density plot showing charge patterns
sns.set_theme(style='whitegrid', palette='muted')

fig, ax = plt.subplots(figsize=(10, 5))
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn',
            fill=True, alpha=0.4, ax=ax)
ax.set_title('Monthly Charges Distribution: Churned vs. Retained Customers',
             fontweight='bold')
ax.set_xlabel('Monthly Charges ($)')
ax.set_ylabel('Density')
plt.tight_layout()
plt.savefig('../output/charts/monthly_charges_distribution.png', dpi=150)
plt.show()

**Visual Insight:** Churned customers cluster in the $70-$100 range, while retained customers show concentration under $30 and between $50-$70. The premium pricing tier ($70+) without sufficient service adoption drives dissatisfaction.

---

## Tenure Lifecycle Analysis: When Do Customers Leave?

### Create Tenure Segments

In [ ]:
# Segment customers by lifecycle stage
df['Tenure_Band'] = pd.cut(df['tenure'],
                            bins=[0, 3, 12, 24, 48, 72],
                            labels=['0-3 months', '4-12 months',
                                    '13-24 months', '25-48 months', '49-72 months'],
                            include_lowest=True)

# Aggregate churn metrics by tenure band
tenure_churn = (df.groupby('Tenure_Band', observed=True)
                  .agg(
                      Customers=('customerID', 'count'),
                      Churned=('Churn_Flag', 'sum'),
                      Churn_Rate=('Churn_Flag', 'mean'),
                      Avg_Monthly=('MonthlyCharges', 'mean')
                  ))

tenure_churn['Churn_Rate_%'] = (tenure_churn['Churn_Rate'] * 100).round(1)
tenure_churn[['Customers', 'Churned', 'Churn_Rate_%', 'Avg_Monthly']].round(2)

**Critical Finding:** First 3 months show 56% churn rate - the highest risk period. Churn drops steadily after 12 months, reaching just 9.5% for customers beyond 4 years.

### Lifecycle Visualization

In [ ]:
# Calculate monthly rolling churn rate
monthly_churn = (df.groupby('tenure')
                   .agg(
                       Total_At_Tenure=('customerID', 'count'),
                       Churned_At_Tenure=('Churn_Flag', 'sum')
                   )
                   .assign(Churn_Rate_At_Tenure=lambda x:
                           x['Churned_At_Tenure'] / x['Total_At_Tenure'])
                   .reset_index())

monthly_churn['Churn_Rate_Rolling'] = (monthly_churn['Churn_Rate_At_Tenure']
                                        .rolling(window=3, center=True)
                                        .mean())

# Create dual visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Churn rate by tenure band
tenure_churn['Churn_Rate'].plot(kind='bar', ax=axes[0],
                                 color='#d73027', edgecolor='white')
axes[0].set_title('Churn Rate by Customer Tenure', fontweight='bold')
axes[0].set_ylabel('Churn Rate')
axes[0].set_xlabel('Tenure Band')
axes[0].tick_params(axis='x', rotation=30)

for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1%}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=10)

# Right: Rolling churn rate across months
axes[1].plot(monthly_churn['tenure'],
             monthly_churn['Churn_Rate_Rolling'],
             color='#4575b4', linewidth=2)
axes[1].fill_between(monthly_churn['tenure'],
                      monthly_churn['Churn_Rate_Rolling'],
                      alpha=0.2, color='#4575b4')
axes[1].set_title('Churn Rate Across Customer Lifecycle\n(3-Month Rolling Average)',
                   fontweight='bold')
axes[1].set_xlabel('Tenure (Months)')
axes[1].set_ylabel('Churn Rate')

plt.tight_layout()
plt.savefig('../output/charts/tenure_cohort_analysis.png', dpi=150)
plt.show()

**Lifecycle Insight:**

The business has an early-stage retention problem. Months 0-12 represent the critical intervention window where retention campaigns matter most. After 12 months, churn drops significantly, suggesting customers who survive onboarding become sticky.

**Implication:** Focus retention budget on first-year customers, particularly those 0-3 months in. Don't waste resources on 25+ month customers (already locked in at 9.5% churn).

---

## Revenue Impact Quantification

### Monthly Recurring Revenue (MRR) Analysis

In [ ]:
# Calculate revenue metrics
total_mrr = df['MonthlyCharges'].sum()
churned_mrr = df[df['Churn'] == 'Yes']['MonthlyCharges'].sum()
retained_mrr = total_mrr - churned_mrr
annual_loss = churned_mrr * 12

# Display metrics
revenue_metrics = pd.DataFrame({
    'Metric': ['Total MRR', 'Retained MRR', 'Lost to Churn (Monthly)', 'Annual Revenue Loss'],
    'Value': [f'${total_mrr:,.0f}', f'${retained_mrr:,.0f}',
              f'${churned_mrr:,.0f}', f'${annual_loss:,.0f}']
})
revenue_metrics

**Business Impact:** $139,131/month lost to churn = **$1.67 million annual revenue exposure**

### Revenue at Risk by Contract Type

In [ ]:
# Calculate revenue exposure by contract
df['Revenue_At_Risk'] = df.apply(lambda row: row['MonthlyCharges'] if row['Churn'] == 'Yes' else 0, axis=1)

contract_revenue = df.groupby('Contract').agg({
    'Revenue_At_Risk': 'sum',
    'MonthlyCharges': 'sum'
}).round(0)

contract_revenue['Pct_At_Risk'] = (contract_revenue['Revenue_At_Risk'] /
                                    contract_revenue['MonthlyCharges'] * 100).round(1)
contract_revenue

**Key Finding:** Month-to-month contracts represent 47% of revenue at risk despite being 55% of customer base. Two-year contracts show only 3% exposure - validates contract conversion as primary retention lever.

### Revenue Impact Visualization

In [ ]:
# MRR waterfall chart
fig, ax = plt.subplots(figsize=(10, 6))

categories = ['Total MRR', 'Retained MRR', 'Lost to Churn']
values = [total_mrr, retained_mrr, churned_mrr]
colors = ['#4575b4', '#91bfdb', '#d73027']

bars = ax.bar(categories, values, color=colors, width=0.6, edgecolor='white', linewidth=2)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1500,
            f'${val:,.0f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('Monthly Recurring Revenue: Retention vs. Churn Impact',
             fontweight='bold', fontsize=14, pad=15)
ax.set_ylabel('Monthly Revenue ($)', fontweight='bold')
ax.set_ylim(0, total_mrr * 1.15)
sns.despine()
plt.tight_layout()
plt.savefig('../output/charts/mrr_waterfall.png', dpi=150)
plt.show()

---

## Risk Segmentation Framework

### Framework Methodology

**Approach:** Rule-based customer scoring using six validated churn predictors. Each customer receives a risk score (0-11 points) based on their profile.

**Risk Factors & Weighting:**
1. **Contract Type** (3 points) - Month-to-month = highest risk
2. **Service Engagement** (2 points) - No TechSupport AND no OnlineSecurity
3. **Tenure** (2 points) - First 12 months = elevated risk
4. **Value Perception** (2 points) - High charges (>$65) with low services (<2)
5. **Payment Method** (1 point) - Electronic check
6. **Paperless Billing** (1 point) - Yes

**Rationale:** Point allocation reflects relative churn rate differentials observed in data. Logic-based approach ensures interpretability for stakeholder communication.

### Risk Factor Validation

In [ ]:
# Calculate churn rates for each risk factor

# Factor 1: Contract (already calculated above)
contract_risk = df.groupby('Contract')['Churn_Flag'].mean() * 100

# Factor 2: Service engagement
df['Low_Service_Engagement'] = ((df['TechSupport'] == 'No') &
                                  (df['OnlineSecurity'] == 'No'))
service_risk = df.groupby('Low_Service_Engagement')['Churn_Flag'].mean() * 100

# Factor 3: Tenure
df['Early_Tenure'] = df['tenure'] <= 12
tenure_risk = df.groupby('Early_Tenure')['Churn_Flag'].mean() * 100

# Factor 4: Value perception
df['High_Cost_Low_Value'] = ((df['MonthlyCharges'] > 65) & (df['Service_Count'] < 2))
value_risk = df.groupby('High_Cost_Low_Value')['Churn_Flag'].mean() * 100

# Factor 5: Payment method
df['Electronic_Check'] = df['PaymentMethod'] == 'Electronic check'
payment_risk = df.groupby('Electronic_Check')['Churn_Flag'].mean() * 100

# Factor 6: Paperless billing
df['Paperless'] = df['PaperlessBilling'] == 'Yes'
paperless_risk = df.groupby('Paperless')['Churn_Flag'].mean() * 100

# Summary table
risk_factors = pd.DataFrame({
    'Risk Factor': [
        'Month-to-month contract',
        'Low service engagement',
        'Early tenure (≤12 months)',
        'High cost, low value',
        'Electronic check payment',
        'Paperless billing'
    ],
    'Churn_Rate_%': [
        contract_risk['Month-to-month'],
        service_risk[True],
        tenure_risk[True],
        value_risk[True],
        payment_risk[True],
        paperless_risk[True]
    ],
    'Points_Assigned': [3, 2, 2, 2, 1, 1]
})

risk_factors['Churn_Rate_%'] = risk_factors['Churn_Rate_%'].round(1)
risk_factors

### Build Risk Scoring Function

In [ ]:
def calculate_risk_score(row):
    """
    Calculate churn risk score (0-11 points).
    Higher scores indicate higher churn risk.
    """
    score = 0

    # Contract type (3 points)
    if row['Contract'] == 'Month-to-month':
        score += 3

    # Service engagement (2 points)
    if row['TechSupport'] == 'No' and row['OnlineSecurity'] == 'No':
        score += 2

    # Tenure (2 points)
    if row['tenure'] <= 12:
        score += 2

    # Value perception (2 points)
    if row['MonthlyCharges'] > 65 and row['Service_Count'] < 2:
        score += 2

    # Payment method (1 point)
    if row['PaymentMethod'] == 'Electronic check':
        score += 1

    # Paperless billing (1 point)
    if row['PaperlessBilling'] == 'Yes':
        score += 1

    return score

# Apply scoring
df['Risk_Score'] = df.apply(calculate_risk_score, axis=1)

# Check distribution
print("Risk Score Distribution:")
print(df['Risk_Score'].value_counts().sort_index())
print(f"\nMean Score: {df['Risk_Score'].mean():.2f}")
print(f"Median Score: {df['Risk_Score'].median():.0f}")

### Create Risk Tiers

In [ ]:
# Segment into risk tiers
df['Risk_Tier'] = pd.cut(
    df['Risk_Score'],
    bins=[-1, 2, 5, 7, 11],
    labels=['Low Risk', 'Moderate Risk', 'High Risk', 'Critical Risk']
)

# Tier distribution
tier_summary = pd.DataFrame({
    'Customers': df['Risk_Tier'].value_counts().sort_index(),
    'Pct_of_Base': (df['Risk_Tier'].value_counts().sort_index() / len(df) * 100).round(1)
})
tier_summary

### Framework Validation

In [ ]:
# Validate that churn rate increases with risk tier
validation = df.groupby('Risk_Tier', observed=True).agg(
    Total_Customers=('customerID', 'count'),
    Churned_Customers=('Churn_Flag', 'sum'),
    Churn_Rate=('Churn_Flag', 'mean'),
    MRR_At_Risk=('Revenue_At_Risk', 'sum')
)

validation['Churn_Rate_%'] = (validation['Churn_Rate'] * 100).round(1)
validation['MRR_At_Risk'] = validation['MRR_At_Risk'].round(0)
validation[['Total_Customers', 'Churned_Customers', 'Churn_Rate_%', 'MRR_At_Risk']]

**Framework Validates:** Churn rate increases monotonically from Low (5.4%) → Moderate (21.6%) → High (40.9%) → Critical (62.9%). Clear tier separation confirms scoring logic identifies true risk.

In [ ]:
# Visualize validation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn rate by tier
axes[0].bar(validation.index, validation['Churn_Rate_%'],
            color=['green', 'yellow', 'orange', 'red'], alpha=0.7)
axes[0].set_title('Churn Rate by Risk Tier', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Risk Tier')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(validation['Churn_Rate_%']):
    axes[0].text(i, v + 1, f'{v}%', ha='center', fontweight='bold')

# Customer distribution
pct_dist = (validation['Total_Customers'] / validation['Total_Customers'].sum() * 100).round(1)
axes[1].bar(validation.index, pct_dist, color='steelblue', alpha=0.7)
axes[1].set_title('Customer Distribution by Risk Tier', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Risk Tier')
axes[1].set_ylabel('% of Customers')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(pct_dist):
    axes[1].text(i, v + 0.5, f'{v}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Export High-Risk Customers for Retention Team

In [ ]:
# Filter High and Critical risk customers
high_risk_customers = df[df['Risk_Tier'].isin(['High Risk', 'Critical Risk'])].copy()

# Create export dataset
export_columns = [
    'customerID', 'Risk_Tier', 'Risk_Score', 'Contract', 'tenure',
    'MonthlyCharges', 'TotalCharges', 'PaymentMethod', 'InternetService',
    'TechSupport', 'OnlineSecurity', 'Service_Count', 'Churn'
]

high_risk_export = high_risk_customers[export_columns].sort_values('Risk_Score', ascending=False)

# Save to CSV
output_file = '../output/high_risk_customers.csv'
high_risk_export.to_csv(output_file, index=False)

print(f"✓ Exported {len(high_risk_export)} high-risk customers to {output_file}")
print(f"\nBreakdown:")
print(high_risk_export['Risk_Tier'].value_counts())
print(f"\nEstimated MRR at Risk: ${high_risk_export['MonthlyCharges'].sum():,.0f}")

---

## Executive-Ready Visualizations

### Chart 1: Contract Type Impact on Churn

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

contract_order = ['Month-to-month', 'One year', 'Two year']
churn_rates_display = df.groupby('Contract')['Churn_Flag'].mean() * 100
churn_rates = [churn_rates_display[c] for c in contract_order]
colors = ['#d73027', '#fc8d59', '#91bfdb']

bars = ax.bar(contract_order, churn_rates, color=colors, edgecolor='white', linewidth=2)

for bar, rate in zip(bars, churn_rates):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{rate:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('Churn Rate by Contract Type', fontweight='bold', fontsize=14, pad=15)
ax.set_ylabel('Churn Rate (%)', fontweight='bold')
ax.set_xlabel('Contract Type', fontweight='bold')
ax.set_ylim(0, max(churn_rates) * 1.15)
sns.despine()
plt.tight_layout()
plt.savefig('../output/charts/contract_churn_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### Chart 2: Monthly Charge Distribution Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

churned = df[df['Churn'] == 'Yes']['MonthlyCharges']
retained = df[df['Churn'] == 'No']['MonthlyCharges']

ax.hist(retained, bins=30, alpha=0.6, label='Retained', color='#91bfdb', edgecolor='white')
ax.hist(churned, bins=30, alpha=0.6, label='Churned', color='#d73027', edgecolor='white')

ax.axvline(retained.median(), color='#4575b4', linestyle='--', linewidth=2,
           label=f'Retained Median: ${retained.median():.2f}')
ax.axvline(churned.median(), color='#a50026', linestyle='--', linewidth=2,
           label=f'Churned Median: ${churned.median():.2f}')

ax.set_title('Monthly Charges: Churned vs. Retained Customers', fontweight='bold', fontsize=14, pad=15)
ax.set_xlabel('Monthly Charges ($)', fontweight='bold')
ax.set_ylabel('Customer Count', fontweight='bold')
ax.legend(frameon=False)
sns.despine()
plt.tight_layout()
plt.savefig('../output/charts/monthly_charge_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### Chart 3: Risk Tier Validation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

tier_order = ['Critical Risk', 'High Risk', 'Moderate Risk', 'Low Risk']
validation_sorted = validation.reindex(tier_order)
churn_rates_tier = validation_sorted['Churn_Rate_%'].values
colors = ['#a50026', '#d73027', '#fc8d59', '#91bfdb']

bars = ax.bar(tier_order, churn_rates_tier, color=colors, edgecolor='white', linewidth=2)

for bar, rate in zip(bars, churn_rates_tier):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{rate:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('Risk Segmentation Validation: Churn Rate by Risk Tier',
             fontweight='bold', fontsize=14, pad=15)
ax.set_ylabel('Actual Churn Rate (%)', fontweight='bold')
ax.set_xlabel('Risk Tier', fontweight='bold')
ax.set_ylim(0, max(churn_rates_tier) * 1.15)
sns.despine()
plt.tight_layout()
plt.savefig('../output/charts/risk_tier_validation.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Executive Summary: Key Findings & Recommendations

### Business Impact

**Revenue Exposure:** Analysis of 7,043 customer records reveals a **26.5% churn rate** translating to **$139,128/month** in lost recurring revenue, or **$1.67 million annualized**.

### Finding 1: Contract Type Dominates Churn Risk

Month-to-month customers churn at **42.7%** (15x higher than two-year contracts at 2.8%). This segment represents the majority of churned revenue despite standard industry pricing.

**Recommendation:** Contract conversion campaign targeting month-to-month customers in High/Critical risk tiers. Even 10% conversion to one-year contracts could preserve $5,000-7,000 monthly revenue.

### Finding 2: First 12 Months Are Critical

Customers in first year churn at **47.4%** compared to **17.1%** after year one. The 0-12 month period represents a critical onboarding and value-discovery window.

**Recommendation:** Implement 30/60/90-day customer health checks for early-tenure customers, especially month-to-month contracts. Prioritize onboarding quality and early value demonstration.

### Finding 3: Value Perception Gap

Customers paying >$65/month with <2 add-on services churn at **57.8%**, indicating severe value misalignment. These customers pay premium rates without sufficient service adoption to justify cost.

**Recommendation:** Targeted bundle promotions to high-cost, low-service customers. Transform retention problem into upsell opportunity by adding TechSupport, OnlineSecurity, or both.

### Risk Segmentation Output

**2,598 customers** flagged as High or Critical risk, representing **$100,668/month** in at-risk MRR. Exported customer list provides actionable targets for retention outreach.

**Framework Validation:**
- Critical Risk: 62.9% churn rate (12x higher than Low Risk)
- High Risk: 40.9% churn rate
- Moderate Risk: 21.6% churn rate
- Low Risk: 5.4% churn rate

### Implementation Priorities

**High Priority:**
- Contract conversion offers for month-to-month customers with Risk Score ≥6
- Early lifecycle intervention at 60-day mark for customers <6 months tenure
- **Est. Impact:** $10-15K monthly revenue preserved, 15-20% reduction in early churn

**Medium Priority:**
- Bundle promotion with discount for high-charge, low-service customers
- **Est. Impact:** Dual retention + upsell opportunity

**Low Priority:**
- Payment method incentives for electronic check users

### Limitations & Future Work

**Data Gaps:** No support ticket history, NPS scores, or usage metrics beyond service adoption. Customer satisfaction data would strengthen causal understanding.

**Analysis Constraints:** Point-in-time snapshot limits lifecycle trend validation. Time-series data would enable cohort-based survival analysis.

**Model Enhancement:** Rule-based scoring prioritizes interpretability. Machine learning approaches (gradient boosting) could improve prediction accuracy at cost of explainability.

### Technical Approach

- **Tools:** Python (Pandas, Scikit-learn), Matplotlib/Seaborn for visualization
- **Model:** Logistic Regression with class balancing (80% recall achieved)
- **Framework:** Rule-based risk scoring (0-11 points) with four-tier segmentation
- **Output:** Actionable customer list, executive visualizations, revenue impact quantification

---

**Analysis Date:** March 2026  
**Dataset:** 7,043 customer records with 21 features  
**Code Repository:** Available for review and reproduction